# Lab 2 - Module 4: Mountain Landscape - Gradient Descent Limitations

**Learning Objectives:**
- Understand how GD gets trapped at local optima
- See the importance of starting position
- Connect to real neural network training challenges
- Recognize when GD fails despite perfect implementation

**Time:** ~15 minutes

---

**IMPORTANT:** Enter the same group code from Lab 1!

## Connection to Previous Work

Previously, you:
- Manually explored a mountain landscape with multiple peaks
- Chose (x, y) locations to sample
- Tried to find the global maximum
- Discovered that finding all peaks was hard!

**Today:** Gradient descent will face the same challenge!

### The Problem:
- GD only sees **local slope** (gradient)
- GD follows the steepest uphill direction
- GD gets **stuck** at the first peak it reaches
- GD cannot "see" that a higher peak exists elsewhere

## 1. Setup: Generate Same Mountain Landscape

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from ipywidgets import FloatText, Button, Checkbox, Output, VBox, HBox
from IPython.display import display

group_code = int(input("Enter your group code: "))
np.random.seed(group_code)

# Skip line and parabola parameters (we only need mountain)
_ = np.random.uniform(-3, 3)  # true_m
_ = np.random.uniform(-5, 5)  # true_b
_ = np.random.uniform(0.5, 2.0)  # hidden_a
_ = np.random.uniform(-4, 4)  # hidden_b
_ = np.random.uniform(-10, 10)  # hidden_c

# Mountain landscape parameters
num_peaks = np.random.randint(3, 6)
peak_centers = []
peak_heights = []
peak_widths = []

# Generate peaks with enforced minimum separation so they don't merge
min_separation = 2.5

for _ in range(num_peaks):
    # Try to place each peak with minimum distance from existing peaks
    for attempt in range(50):
        cx = np.random.uniform(-3.5, 3.5)
        cy = np.random.uniform(-3.5, 3.5)
        # Check distance to all existing peaks
        too_close = False
        for (ex, ey) in peak_centers:
            if np.sqrt((cx - ex)**2 + (cy - ey)**2) < min_separation:
                too_close = True
                break
        if not too_close:
            break
    # If we couldn't find a separated spot after 50 tries, use the last random one anyway
    
    height = np.random.uniform(2.0, 5.0)
    width = np.random.uniform(0.8, 1.5)
    peak_centers.append((cx, cy))
    peak_heights.append(height)
    peak_widths.append(width)

def mountain_height(pos):
    """Mountain landscape with multiple Gaussian peaks.
    
    Args:
        pos: [x, y] array or scalars
    
    Returns:
        Altitude (scalar or array)
    """
    if isinstance(pos, (list, tuple)):
        pos = np.array(pos)
    
    x = pos[..., 0] if pos.ndim > 1 else pos[0]
    y = pos[..., 1] if pos.ndim > 1 else pos[1]
    
    z = np.zeros_like(x, dtype=float)
    for (cx, cy), h, w in zip(peak_centers, peak_heights, peak_widths):
        z += h * np.exp(-(((x - cx)**2 + (y - cy)**2) / (2 * w**2)))
    return z

# Find global maximum
grid_size = 100
x_vals = np.linspace(-5, 5, grid_size)
y_vals = np.linspace(-5, 5, grid_size)
Xg, Yg = np.meshgrid(x_vals, y_vals)
Zg = np.zeros_like(Xg)
for i in range(grid_size):
    for j in range(grid_size):
        Zg[i, j] = mountain_height([Xg[i, j], Yg[i, j]])

flat_idx = np.argmax(Zg)
i_max, j_max = np.unravel_index(flat_idx, Zg.shape)
x_global = Xg[i_max, j_max]
y_global = Yg[i_max, j_max]
h_global = Zg[i_max, j_max]

print(f"✓ Mountain landscape loaded")
print(f"Number of peaks: {num_peaks}")
for i, ((cx, cy), h, w) in enumerate(zip(peak_centers, peak_heights, peak_widths)):
    print(f"  Peak {i+1}: center=({cx:.2f}, {cy:.2f}), height={h:.2f}")
print(f"\nGlobal maximum at: ({x_global:.2f}, {y_global:.2f}) with height {h_global:.2f}")
print("(Revealed for learning purposes)")

## 2. Gradient Ascent (Uphill Climbing)

To find peaks (maxima), we'll use **gradient ascent** instead of descent:

```
new = old + learning_rate × gradient  (note: + instead of -)
```

This moves **uphill** toward local peaks.

In [ ]:
def compute_gradient_mountain(pos, h=1e-4):
    """Compute numerical gradient of mountain height.
    
    Args:
        pos: [x, y] position
    
    Returns:
        [grad_x, grad_y]
    """
    pos = np.array(pos, dtype=float)
    grad = np.zeros(2)
    
    for i in range(2):
        pos_forward = pos.copy()
        pos_backward = pos.copy()
        pos_forward[i] += h
        pos_backward[i] -= h
        grad[i] = (mountain_height(pos_forward) - mountain_height(pos_backward)) / (2 * h)
    
    return grad

def gradient_ascent_step(pos, learning_rate):
    """One step of gradient ascent (climbing uphill)."""
    grad = compute_gradient_mountain(pos)
    return pos + learning_rate * grad  # + for ascent (uphill)

def run_gradient_ascent(start_pos, learning_rate, max_steps=50, tol=1e-4):
    """Run gradient ascent from starting position."""
    history = {
        'pos': [np.array(start_pos, dtype=float)],
        'height': [mountain_height(start_pos)],
        'grad': [],
        'converged': False,
        'n_steps': 0
    }
    
    pos_current = np.array(start_pos, dtype=float)
    
    for step in range(max_steps):
        grad = compute_gradient_mountain(pos_current)
        history['grad'].append(grad)
        
        # Ascent step
        pos_new = gradient_ascent_step(pos_current, learning_rate)
        h_new = mountain_height(pos_new)
        
        history['pos'].append(pos_new.copy())
        history['height'].append(h_new)
        history['n_steps'] = step + 1
        
        # Check convergence (gradient near zero)
        if np.linalg.norm(grad) < tol:
            history['converged'] = True
            break
        
        pos_current = pos_new
    
    return history

print("✓ Gradient ascent functions ready")

## 3. Prediction Questions (Answer BEFORE running)

**Q10 (PREDICTION):** 

Think about starting positions:
- Starting at (1, 1): Will gradient ascent find the global maximum? Why or why not?
- Will different starting points reach different peaks?

Write your predictions on the answer sheet!

## 4. Interactive: Run GD from Different Starting Points

In [ ]:
# State for multiple GD runs
gd_runs = []
colors_runs = ['red', 'blue', 'green', 'orange', 'purple', 'brown']

# Precompute contour levels that make all peaks visible
# Use levels proportional to sqrt so low-altitude structure is visible
max_z = Zg.max()
contour_levels = np.concatenate([
    np.linspace(0, 0.3 * max_z, 10),
    np.linspace(0.3 * max_z, max_z, 20)
])
contour_levels = np.unique(contour_levels)

# Widgets
x_start_input = FloatText(description="Start x:", value=0.0, step=0.5)
y_start_input = FloatText(description="Start y:", value=0.0, step=0.5)
lr_mountain_input = FloatText(description="Learning rate:", value=0.5, step=0.1)
run_ga_button = Button(description="Run Gradient Ascent", button_style='success')
reset_mountain_button = Button(description="Reset All", button_style='warning')
output_mountain = Output()

def plot_mountain_results():
    """Plot all GD runs on mountain landscape."""
    with output_mountain:
        output_mountain.clear_output(wait=True)

        # Create plot
        fig, ax = plt.subplots(figsize=(10, 8), dpi=100)

        # Always show the full landscape so all peaks are visible
        contour = ax.contourf(Xg, Yg, Zg, levels=contour_levels, cmap='terrain', alpha=0.7)
        # Add contour lines for extra clarity
        ax.contour(Xg, Yg, Zg, levels=contour_levels[::3], colors='black', alpha=0.2, linewidths=0.5)
        plt.colorbar(contour, ax=ax, label='Altitude')

        # Mark all peak centers
        for i, ((cx, cy), h) in enumerate(zip(peak_centers, peak_heights)):
            if abs(h - max(peak_heights)) < 0.01:
                # Global max gets special marker
                ax.scatter([cx], [cy], c='yellow', marker='*',
                          s=500, edgecolors='black', linewidths=3, zorder=20, 
                          label='Global maximum' if i == 0 or not any(abs(ph - max(peak_heights)) < 0.01 for ph in peak_heights[:i]) else None)
            else:
                ax.scatter([cx], [cy], c='white', marker='^',
                          s=150, edgecolors='black', linewidths=2, zorder=19,
                          label='Local peak' if i == 0 or all(abs(ph - max(peak_heights)) < 0.01 for ph in peak_heights[:i]) else None)

        # Plot each GD run
        for i, run in enumerate(gd_runs):
            pos_hist = np.array(run['history']['pos'])
            color = colors_runs[i % len(colors_runs)]

            # Path
            ax.plot(pos_hist[:, 0], pos_hist[:, 1], 'o-',
                   color=color, linewidth=2, markersize=5, alpha=0.8,
                   label=f"Run {i+1}: start ({run['start'][0]:.1f}, {run['start'][1]:.1f})")

            # Mark start
            ax.scatter([pos_hist[0, 0]], [pos_hist[0, 1]],
                      c='white', marker='o', s=150, edgecolors=color, linewidths=3, zorder=10)

            # Mark end (peak reached)
            ax.scatter([pos_hist[-1, 0]], [pos_hist[-1, 1]],
                      c=color, marker='*', s=300, edgecolors='black', linewidths=2, zorder=11)

        ax.set_xlabel('x', fontsize=12)
        ax.set_ylabel('y', fontsize=12)
        ax.set_title(f'Gradient Ascent on Mountain Landscape ({num_peaks} peaks)', fontsize=14, fontweight='bold')
        ax.set_xlim(-5, 5)
        ax.set_ylim(-5, 5)
        ax.set_aspect('equal')
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=9, loc='best')

        plt.tight_layout()
        plt.show()

        # Summary table
        if gd_runs:
            print("\nSummary of Gradient Ascent Runs:")
            print("="*80)
            summary_data = []
            for i, run in enumerate(gd_runs):
                final_pos = run['history']['pos'][-1]
                final_h = run['history']['height'][-1]
                summary_data.append({
                    'Run': i + 1,
                    'Start (x, y)': f"({run['start'][0]:.2f}, {run['start'][1]:.2f})",
                    'Final (x, y)': f"({final_pos[0]:.2f}, {final_pos[1]:.2f})",
                    'Final height': f"{final_h:.3f}",
                    'Steps': run['history']['n_steps'],
                    'Found global?': 'Yes' if abs(final_h - h_global) < 0.5 else 'No'
                })

            display(pd.DataFrame(summary_data))
            print(f"\nGlobal maximum: ({x_global:.2f}, {y_global:.2f}), height = {h_global:.2f}")
            print(f"\nNotice: Different starting points lead to different local peaks!")

def on_run_ga_click(b):
    x_start = x_start_input.value
    y_start = y_start_input.value
    lr = lr_mountain_input.value

    # Run gradient ascent
    history = run_gradient_ascent([x_start, y_start], lr, max_steps=50)

    gd_runs.append({
        'start': [x_start, y_start],
        'lr': lr,
        'history': history
    })

    plot_mountain_results()

def on_reset_click(b):
    global gd_runs
    gd_runs = []
    with output_mountain:
        output_mountain.clear_output()
        print("Reset! Try new starting positions.")

run_ga_button.on_click(on_run_ga_click)
reset_mountain_button.on_click(on_reset_click)

# Show the landscape immediately so students can see all peaks
plot_mountain_results()

print("\nInteractive Gradient Ascent on Mountain Landscape")
print("="*80)
print("1. Enter starting (x, y) position (range: -5 to 5)")
print("2. Set learning rate (try 0.5)")
print("3. Click 'Run Gradient Ascent' to see GD climb to nearest peak")
print("4. Try MULTIPLE starting points to see different outcomes\n")
print("Suggested starting points to try:")
print("  - (0, 0)")
print("  - (2, 2)")
print("  - (-3, 1)")
print("  - (1, -2)")
print("="*80)

display(VBox([
    HBox([x_start_input, y_start_input, lr_mountain_input]),
    HBox([run_ga_button, reset_mountain_button]),
    output_mountain
]))

## Questions for Your Answer Sheet

**Q11.** Based on your experiments:
- Did gradient ascent find the global maximum from every starting point?
- Why can't GD "see" distant peaks?
- What strategies might help overcome this limitation?

## Key Takeaways: Limitations of Gradient Descent

### What We Learned:

1. **Local Optima Problem:**
   - GD only sees **local slope**, not the full landscape
   - Gets trapped at the first peak/valley it reaches
   - Cannot escape local optima on its own

2. **Starting Point Matters:**
   - Different starting points → different local optima
   - No way to know if you found global optimum
   - In ML: Weight initialization is crucial!

3. **Gradient Descent is "Greedy":**
   - Always follows steepest descent/ascent
   - Never "backtracks" or explores
   - Deterministic path from starting point

### Real Machine Learning Solutions:

1. **Random Restarts:** Try multiple starting points
2. **Momentum:** Add "inertia" to push through small barriers
3. **Stochastic GD:** Add noise to escape local minima
4. **Adaptive Methods:** Adjust learning rate dynamically (Adam, RMSprop)
5. **Good Initialization:** Smart starting points (Xavier, He initialization)
6. **Acceptance of Local Minima:** In practice, "good enough" local minima work!

### Why Neural Networks Still Work:

- High-dimensional spaces have many "good" local minima
- Most local minima are close in performance to global minimum
- Overparameterization helps: many paths to good solutions
- Modern architectures and techniques reduce the problem

## Lab 2 Complete!

Congratulations! You've completed Lab 2 and learned:

- The universal update rule: `new = old - learning_rate x gradient`
- How gradient descent automates parameter search
- The critical importance of learning rate (Goldilocks problem)
- How GD navigates parameter space systematically
- The fundamental limitation: getting stuck at local optima
- Why starting position matters enormously

### Next Steps:

1. **Answer Q10 and Q11** on your answer sheet
2. **Review your predictions** - How accurate were they?
3. **Return to the LMS** to submit your work